In [14]:
# M24CA1L306 Data Science Lab - Lab Exercise #05
# Student: Abhinav | MAC25MCA-2002 | S3 MCA (2025-27)
# Dataset: Breast_Cancer.csv

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

matplotlib.use('Agg')

df = pd.read_csv("../Datasets/Breast_Cancer.csv")
df.drop(columns=["id"], inplace=True)

mean_features = [c for c in df.columns if c.endswith("_mean")]
worst_features = [c for c in df.columns if c.endswith("_worst")]
se_features = [c for c in df.columns if c.endswith("_se")]

print(df.shape)
print(df.head())
print(df["diagnosis"].value_counts())
print(df.isnull().sum())

(569, 31)
  diagnosis  radius_mean  texture_mean  perimeter_mean  area_mean  \
0         M        17.99         10.38          122.80     1001.0   
1         M        20.57         17.77          132.90     1326.0   
2         M        19.69         21.25          130.00     1203.0   
3         M        11.42         20.38           77.58      386.1   
4         M        20.29         14.34          135.10     1297.0   

   smoothness_mean  compactness_mean  concavity_mean  concave points_mean  \
0          0.11840           0.27760          0.3001              0.14710   
1          0.08474           0.07864          0.0869              0.07017   
2          0.10960           0.15990          0.1974              0.12790   
3          0.14250           0.28390          0.2414              0.10520   
4          0.10030           0.13280          0.1980              0.10430   

   symmetry_mean  ...  radius_worst  texture_worst  perimeter_worst  \
0         0.2419  ...         25.38      

In [3]:
# ─────────────────────────────────────────────
# 1. Distribution of Malignant vs Benign Cases
# ─────────────────────────────────────────────
counts = df["diagnosis"].value_counts()
pct = df["diagnosis"].value_counts(normalize=True) * 100

plt.figure(figsize=(6, 5))
bars = plt.bar(["Benign (B)", "Malignant (M)"], counts,
               color=["steelblue", "tomato"], edgecolor="black")
for bar, p in zip(bars, [pct["B"], pct["M"]]):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 3,
             f"{p:.1f}%", ha="center", fontsize=11)
plt.title("Distribution of Malignant vs Benign Cases")
plt.ylabel("Number of Cases")
plt.tight_layout()
plt.savefig("../Outputs/Lab-5_Output/lab05_01_diagnosis_distribution.png", dpi=100)
plt.close()

# Findings: The dataset contains 357 benign (62.7%) and 212 malignant
# (37.3%) cases. The class imbalance is moderate and should be considered
# during classification model training.

In [4]:
# ─────────────────────────────────────────────
# 2. Feature Differences - Boxplots (radius, texture, area mean)
# ─────────────────────────────────────────────
key_features = ["radius_mean", "texture_mean", "area_mean"]

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, feature in zip(axes, key_features):
    df.boxplot(column=feature, by="diagnosis", ax=ax,
               boxprops=dict(color="black"),
               medianprops=dict(color="red"))
    ax.set_title(feature)
    ax.set_xlabel("Diagnosis")
    ax.set_ylabel(feature)
plt.suptitle("Key Feature Distributions by Diagnosis")
plt.tight_layout()
plt.savefig("../Outputs/Lab-5_Output/lab05_02_key_features_boxplot.png", dpi=100)
plt.close()

# Findings: Malignant tumors show significantly higher radius_mean,
# area_mean and texture_mean compared to benign ones. area_mean shows
# the largest separation, making it a strong candidate for classification.

In [5]:
# ─────────────────────────────────────────────
# 3. Correlation - Radius, Perimeter, Area (Heatmap)
# ─────────────────────────────────────────────
rpa_features = ["radius_mean", "perimeter_mean", "area_mean",
                "radius_worst", "perimeter_worst", "area_worst"]

plt.figure(figsize=(8, 6))
sns.heatmap(df[rpa_features].corr(), annot=True, fmt=".2f",
            cmap="coolwarm", square=True)
plt.title("Correlation - Radius, Perimeter and Area Features")
plt.tight_layout()
plt.savefig("../Outputs/Lab-5_Output/lab05_03_rpa_correlation_heatmap.png", dpi=100)
plt.close()

# Findings: Radius, perimeter and area (both mean and worst) are almost
# perfectly correlated (r > 0.99). This confirms severe multicollinearity
# among size-related features - only one of these groups needs to be
# retained for modelling.

In [6]:
# ─────────────────────────────────────────────
# 4. Violin Plots - concavity_mean and smoothness_mean
# ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, feature in zip(axes, ["concavity_mean", "smoothness_mean"]):
    sns.violinplot(data=df, x="diagnosis", y=feature,
                   palette={"B": "steelblue", "M": "tomato"},
                   inner="quartile", ax=ax)
    ax.set_title(f"Violin Plot - {feature}")
    ax.set_xlabel("Diagnosis (B = Benign, M = Malignant)")
    ax.set_ylabel(feature)
plt.suptitle("Feature Distributions by Diagnosis - Violin Plots")
plt.tight_layout()
plt.savefig("../Outputs/Lab-5_Output/lab05_04_violin_concavity_smoothness.png", dpi=100)
plt.close()

# Findings: concavity_mean shows a much wider and higher distribution
# for malignant tumors, making it a strong discriminating feature.
# smoothness_mean shows moderate separation with considerable overlap,
# suggesting it is a weaker standalone classifier.

C:\Users\abhinav\AppData\Local\Temp\ipykernel_14332\1419575737.py:6: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=df, x="diagnosis", y=feature,
C:\Users\abhinav\AppData\Local\Temp\ipykernel_14332\1419575737.py:6: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=df, x="diagnosis", y=feature,


In [7]:
# ─────────────────────────────────────────────
# 5. Mean vs Worst Feature Distributions
# ─────────────────────────────────────────────
compare_pairs = [("radius_mean", "radius_worst"),
                 ("concavity_mean", "concavity_worst"),
                 ("texture_mean", "texture_worst")]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for col, (mean_f, worst_f) in enumerate(compare_pairs):
    for row, (feature, label) in enumerate([(mean_f, "Mean"), (worst_f, "Worst")]):
        ax = axes[row][col]
        for diag, color in [("B", "steelblue"), ("M", "tomato")]:
            df[df["diagnosis"] == diag][feature].plot.kde(ax=ax, label=diag, color=color)
        ax.set_title(f"{label} - {feature.split('_')[0].capitalize()}")
        ax.set_xlabel(feature)
        ax.legend(fontsize=8)
plt.suptitle("Mean vs Worst Feature Distributions by Diagnosis")
plt.tight_layout()
plt.savefig("../Outputs/Lab-5_Output/lab05_05_mean_vs_worst.png", dpi=100)
plt.close()

# Findings: Worst features consistently show wider separation between
# malignant and benign classes compared to their mean counterparts.
# This suggests worst features carry stronger classification signal
# and may be more useful as model inputs.

In [8]:
# ─────────────────────────────────────────────
# 6. Multicollinearity - Full Correlation Heatmap
# ─────────────────────────────────────────────
plt.figure(figsize=(18, 15))
sns.heatmap(df[mean_features + worst_features].corr(), annot=True, fmt=".1f",
            cmap="coolwarm", square=True, annot_kws={"size": 7})
plt.title("Full Correlation Heatmap - Mean and Worst Features")
plt.tight_layout()
plt.savefig("../Outputs/Lab-5_Output/lab05_06_full_correlation_heatmap.png", dpi=100)
plt.close()

# Findings: Several feature clusters are highly correlated (r > 0.9):
# radius/perimeter/area (mean and worst), and concavity/concave_points.
# These groups are redundant and contribute to multicollinearity.
# Dimensionality reduction or feature selection is recommended before
# training classifiers.

In [9]:
# ─────────────────────────────────────────────
# 7. Top 5 Separating Features - Pairplot
# ─────────────────────────────────────────────
top5 = ["radius_mean", "concavity_mean", "area_mean",
        "concave points_mean", "texture_mean"]

pair = sns.pairplot(df[top5 + ["diagnosis"]], hue="diagnosis",
                    palette={"B": "steelblue", "M": "tomato"},
                    diag_kind="kde", plot_kws={"alpha": 0.5})
pair.fig.suptitle("Pairplot - Top 5 Separating Features", y=1.02)
pair.fig.savefig("../Outputs/Lab-5_Output/lab05_07_top5_pairplot.png", dpi=100)
plt.close()

# Findings: radius_mean, area_mean and concave points_mean show the
# clearest separation between classes in scatter plots. KDE diagonals
# confirm distinct distributions for malignant vs benign across all
# five features. concavity_mean and concave points_mean are highly
# correlated with each other.

In [10]:
# ─────────────────────────────────────────────
# 8. Average Tumor Size - Grouped Bar Plot
# ─────────────────────────────────────────────
size_features = ["radius_mean", "area_mean"]
avg_size = df.groupby("diagnosis")[size_features].mean()

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, feature in zip(axes, size_features):
    bars = ax.bar(["Benign", "Malignant"], avg_size[feature],
                  color=["steelblue", "tomato"], edgecolor="black")
    for bar, val in zip(bars, avg_size[feature]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f"{val:.2f}", ha="center", fontsize=10)
    ax.set_title(f"Average {feature}")
    ax.set_ylabel("Value")
plt.suptitle("Average Tumor Size by Diagnosis")
plt.tight_layout()
plt.savefig("../Outputs/Lab-5_Output/lab05_08_avg_tumor_size.png", dpi=100)
plt.close()

# Findings: Malignant tumors have a considerably larger average radius
# (~17.5 vs ~12.1) and area (~978 vs ~463) compared to benign tumors.
# Tumor size is a strong indicator of malignancy.

In [11]:
# ─────────────────────────────────────────────
# 9. Outlier Detection - area_mean and radius_mean
# ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, feature in zip(axes, ["area_mean", "radius_mean"]):
    df.boxplot(column=feature, by="diagnosis", ax=ax,
               boxprops=dict(color="black"),
               medianprops=dict(color="red"),
               flierprops=dict(marker="o", color="orange", markersize=5))
    ax.set_title(feature)
    ax.set_xlabel("Diagnosis")
    ax.set_ylabel(feature)
plt.suptitle("Outlier Detection - Area and Radius by Diagnosis")
plt.tight_layout()
plt.savefig("../Outputs/Lab-5_Output/lab05_09_outliers_boxplot.png", dpi=100)
plt.close()

# Findings: Both area_mean and radius_mean contain outliers, particularly
# in the malignant class. These extreme values represent unusually large
# tumors and could skew distance-based classifiers like kNN. Outlier
# treatment or robust scaling should be considered before modelling.

In [13]:
# ─────────────────────────────────────────────
# 10. Feature Variance by Class
# ─────────────────────────────────────────────
variance_M = df[df["diagnosis"] == "M"][mean_features].var()
variance_B = df[df["diagnosis"] == "B"][mean_features].var()

var_df = pd.DataFrame({"Malignant": variance_M, "Benign": variance_B})
var_df = var_df.sort_values("Malignant", ascending=False)

var_df.plot(kind="bar", figsize=(14, 6), color=["tomato", "steelblue"], edgecolor="black")
plt.title("Feature Variance by Diagnosis Class (Mean Features)")
plt.ylabel("Variance")
plt.xlabel("Feature")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("../Outputs/Lab-5_Output/lab05_10_feature_variance.png", dpi=100)
plt.close()

# Findings: area_mean has the highest variance by far, especially in the
# malignant class, reflecting wide spread in tumor sizes. Features with
# high variance in the malignant class (area, perimeter, radius) tend to
# be stronger classifiers. Low-variance features like fractal_dimension_mean
# and smoothness_mean contribute less to class separation.